# FAST Adversarial Training: ResNet-50

 This Notebook uses random-start FGSM (FAST), mixed precision, OneCycleLR, clean/PGD-10 validation.

In [ ]:
# Edit these values before starting training.
DATA_PATH = "/kaggle/input/datasets/saraghanbari/trainds"  
OUTPUT_PATH = "/kaggle/working/model.pt"

EPOCHS = 30
BATCH_SIZE = 256  # Total batch size split across both GPUs.
WORKERS = 4
MAX_LR = 0.20
WEIGHT_DECAY = 5e-4
EPSILON = 8 / 255
FGSM_STEP = 10 / 255
VAL_FRACTION = 0.10
SEED = 42

print("Configuration loaded")

Configuration loaded


In [2]:
"""FAST adversarial training for the robustness assignment on Kaggle (2 GPUs)."""

import os
import random
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.models import resnet50
from torchvision.transforms import functional as TF


NUM_CLASSES = 9


def parse_args():
    # The notebook configuration is defined in the cell above.
    return SimpleNamespace(
        data=DATA_PATH, output=OUTPUT_PATH, epochs=EPOCHS,
        batch_size=BATCH_SIZE, workers=WORKERS, lr=MAX_LR,
        weight_decay=WEIGHT_DECAY, epsilon=EPSILON,
        fgsm_step=FGSM_STEP, val_fraction=VAL_FRACTION, seed=SEED,
    )


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class NpzDataset(Dataset):
    def __init__(self, path, train):
        archive = np.load(path)
        self.images = archive["images"]
        self.labels = archive["labels"].astype(np.int64)
        self.train = train

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image = torch.from_numpy(self.images[index]).float().div_(255.0)
        label = int(self.labels[index])
        if self.train:
            image = TF.pad(image, [4, 4, 4, 4], padding_mode="reflect")
            top = torch.randint(0, 9, ()).item()
            left = torch.randint(0, 9, ()).item()
            image = image[:, top : top + 32, left : left + 32]
            if torch.rand(()) < 0.5:
                image = TF.hflip(image)
        return image, label


def stratified_split(labels, val_fraction, seed):
    generator = np.random.default_rng(seed)
    train_indices, val_indices = [], []
    for class_id in range(NUM_CLASSES):
        indices = np.flatnonzero(labels == class_id)
        generator.shuffle(indices)
        n_val = max(1, round(len(indices) * val_fraction))
        val_indices.extend(indices[:n_val].tolist())
        train_indices.extend(indices[n_val:].tolist())
    generator.shuffle(train_indices)
    generator.shuffle(val_indices)
    return train_indices, val_indices


def random_start_fgsm(model, images, labels, epsilon, step, criterion):
    # Random initialization is the key addition that makes single-step FAST
    # training much harder to exploit than plain FGSM training.
    delta = torch.empty_like(images).uniform_(-epsilon, epsilon)
    delta = torch.clamp(images + delta, 0.0, 1.0) - images
    delta.requires_grad_(True)

    with torch.amp.autocast("cuda", enabled=False):
        loss = criterion(model((images + delta).float()), labels)
    gradient = torch.autograd.grad(loss, delta, only_inputs=True)[0]
    delta = delta.detach() + step * gradient.sign()
    delta = torch.clamp(delta, -epsilon, epsilon)
    return torch.clamp(images + delta, 0.0, 1.0).detach()


def pgd_attack(model, images, labels, epsilon, step, steps, criterion):
    delta = torch.empty_like(images).uniform_(-epsilon, epsilon)
    delta = torch.clamp(images + delta, 0.0, 1.0) - images
    for _ in range(steps):
        delta.requires_grad_(True)
        with torch.amp.autocast("cuda", enabled=False):
            loss = criterion(model((images + delta).float()), labels)
        gradient = torch.autograd.grad(loss, delta, only_inputs=True)[0]
        delta = delta.detach() + step * gradient.sign()
        delta = torch.clamp(delta, -epsilon, epsilon)
        delta = torch.clamp(images + delta, 0.0, 1.0) - images
    return torch.clamp(images + delta, 0.0, 1.0).detach()


def evaluate(model, loader, device, criterion, epsilon, max_pgd_batches=10):
    model.eval()
    clean_correct = robust_correct = clean_total = robust_total = 0
    for batch_index, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.no_grad(), torch.amp.autocast("cuda"):
            clean_correct += (model(images).argmax(1) == labels).sum().item()
        clean_total += labels.size(0)

        if batch_index < max_pgd_batches:
            adversarial = pgd_attack(
                model, images, labels, epsilon, step=2 / 255, steps=10,
                criterion=criterion,
            )
            with torch.no_grad(), torch.amp.autocast("cuda"):
                robust_correct += (model(adversarial).argmax(1) == labels).sum().item()
            robust_total += labels.size(0)

    return clean_correct / clean_total, robust_correct / robust_total


def main():
    args = parse_args()
    seed_everything(args.seed)
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for this training script.")
    gpu_count = torch.cuda.device_count()
    if gpu_count < 2:
        raise RuntimeError(
            f"Found {gpu_count} GPU(s). In Kaggle Settings, select the accelerator "
            "option that provides 2 GPUs."
        )
    print("Using GPUs:", [torch.cuda.get_device_name(i) for i in (0, 1)])

    data_path = Path(args.data)
    if data_path.is_dir():
        # Kaggle dataset paths commonly point to the attachment directory.
        matches = list(data_path.rglob("train.npz"))
        if not matches:
            matches = list(data_path.rglob("*.npz"))
    elif data_path.is_file():
        matches = [data_path]
    else:
        matches = list(Path("/kaggle/input").rglob("train.npz"))

    if len(matches) != 1:
        raise FileNotFoundError(
            "Expected exactly one .npz training file. "
            f"Set DATA_PATH to the file or its containing directory. Found: {matches}"
        )
    data_path = matches[0]
    print("Loading", data_path)

    train_data = NpzDataset(data_path, train=True)
    val_data = NpzDataset(data_path, train=False)
    train_indices, val_indices = stratified_split(
        train_data.labels, args.val_fraction, args.seed
    )
    train_loader = DataLoader(
        Subset(train_data, train_indices),
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.workers,
        pin_memory=True,
        persistent_workers=args.workers > 0,
        drop_last=True,
    )
    val_loader = DataLoader(
        Subset(val_data, val_indices),
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.workers,
        pin_memory=True,
        persistent_workers=args.workers > 0,
    )

    base_model = resnet50(weights=None)
    base_model.fc = nn.Linear(base_model.fc.in_features, NUM_CLASSES)
    base_model.cuda(0)
    model = nn.DataParallel(base_model, device_ids=[0, 1], output_device=0)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(
        model.parameters(), lr=args.lr, momentum=0.9,
        weight_decay=args.weight_decay, nesterov=True,
    )
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=args.lr,
        epochs=args.epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.4,
        anneal_strategy="linear",
        cycle_momentum=False,
    )
    scaler = torch.amp.GradScaler("cuda")

    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    best_score = -1.0

    for epoch in range(1, args.epochs + 1):
        model.train()
        running_loss = correct = total = 0
        for images, labels in train_loader:
            images = images.cuda(0, non_blocking=True)
            labels = labels.cuda(0, non_blocking=True)

            # Generate the perturbation in FP32, then train on it with AMP.
            adversarial = random_start_fgsm(
                model, images, labels, args.epsilon, args.fgsm_step, criterion
            )
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                logits = model(adversarial)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)

        clean_acc, pgd10_acc = evaluate(
            model, val_loader, torch.device("cuda:0"), criterion, args.epsilon
        )
        unified_proxy = 0.5 * (clean_acc + pgd10_acc)
        print(
            f"Epoch {epoch:02d}/{args.epochs} | "
            f"loss {running_loss / total:.4f} | train adv {correct / total:.4%} | "
            f"val clean {clean_acc:.4%} | val PGD-10 {pgd10_acc:.4%} | "
            f"proxy {unified_proxy:.4%}"
        )

        # Always unwrap DataParallel before saving: the server expects keys such
        # as 'conv1.weight', not 'module.conv1.weight'.
        if unified_proxy > best_score:
            best_score = unified_proxy
            state = {key: value.detach().cpu() for key, value in model.module.state_dict().items()}
            torch.save(state, output_path)
            print(f"Saved best state dict to {output_path}")

    check_model = resnet50(weights=None)
    check_model.fc = nn.Linear(check_model.fc.in_features, NUM_CLASSES)
    check_model.load_state_dict(torch.load(output_path, map_location="cpu", weights_only=True))
    check_model.eval()
    with torch.no_grad():
        assert check_model(torch.zeros(1, 3, 32, 32)).shape == (1, NUM_CLASSES)
    print(f"Submission check passed. Best validation proxy: {best_score:.4%}")


In [3]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for gpu_id in range(torch.cuda.device_count()):
    print(f"GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)}")

assert torch.cuda.device_count() >= 2, (
    "Select Kaggle Settings > Accelerator > GPU T4 x2, then restart."
)

PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## Train

The best checkpoint uses an equal-weight proxy of clean and PGD-10 validation accuracy.

In [4]:
# Perform training and save the best checkpoint.
os.environ.setdefault("OMP_NUM_THREADS", "4")
main()

Using GPUs: ['Tesla T4', 'Tesla T4']
Loading /kaggle/input/datasets/saraghanbari/trainds/train.npz


/tmp/ipykernel_23/2681082677.py:225: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 01/30 | loss 2.2713 | train adv 25.8638% | val clean 38.9000% | val PGD-10 27.0312% | proxy 32.9656%
Saved best state dict to /kaggle/working/model.pt
Epoch 02/30 | loss 1.7291 | train adv 31.2969% | val clean 49.9600% | val PGD-10 25.9766% | proxy 37.9683%
Saved best state dict to /kaggle/working/model.pt
Epoch 03/30 | loss 1.7405 | train adv 32.5156% | val clean 49.1600% | val PGD-10 27.9297% | proxy 38.5448%
Saved best state dict to /kaggle/working/model.pt
Epoch 04/30 | loss 1.6677 | train adv 34.6763% | val clean 52.1200% | val PGD-10 28.9844% | proxy 40.5522%
Saved best state dict to /kaggle/working/model.pt
Epoch 05/30 | loss 1.6021 | train adv 36.2723% | val clean 45.5200% | val PGD-10 26.2891% | proxy 35.9045%
Epoch 06/30 | loss 1.5764 | train adv 37.4129% | val clean 49.8200% | val PGD-10 28.7109% | proxy 39.2655%
Epoch 07/30 | loss 1.5619 | train adv 37.7679% | val clean 55.2200% | val PGD-10 31.9531% | proxy 43.5866%
Saved best state dict to /kaggle/working/model.pt
E

## Validate and download

Strictly reload the state dictionary into a plain torchvision ResNet-50 and expose model.pt as a download link.

In [ ]:

checkpoint = Path(OUTPUT_PATH)
assert checkpoint.exists(), f"Missing checkpoint: {checkpoint}"

submission_model = resnet50(weights=None)
submission_model.fc = nn.Linear(submission_model.fc.in_features, NUM_CLASSES)
state_dict = torch.load(checkpoint, map_location="cpu", weights_only=True)
submission_model.load_state_dict(state_dict, strict=True)
assert not any(key.startswith("module.") for key in state_dict)

submission_model.eval()
with torch.no_grad():
    assert submission_model(torch.zeros(1, 3, 32, 32)).shape == (1, 9)

print(f"Ready to submit: {checkpoint}")
print(f"Checkpoint size: {checkpoint.stat().st_size / 1024**2:.1f} MiB")

from IPython.display import FileLink, display
display(FileLink(str(checkpoint)))

Ready to submit: /kaggle/working/model.pt
Checkpoint size: 90.0 MiB


/kaggle/working/model.pt